In [ ]:
base_folder = None
save_folder = None
table_path = None
sr = None
height = None
width = None
duration = None
fmin = None
fmax = None

In [ ]:
# if no arguments specified
if base_folder is None:
    base_folder = '/nfs/turbo/umor-sethtem/acoustics-data'
if save_folder is None:
    save_folder = '/nfs/turbo/umor-sethtem/spectrogram-data'
if table_path is None:
    table_path = '../data/combined.split.5s.2d.csv'
if sr is None:
    sr = 32000
if height is None:
    height = 64
if width is None:
    width = 224
if duration is None:
    duration = 5
if fmin is None:
    fmin = 0
if fmax is None:
    fmax = 16000

In [ ]:
# common imports
import os
import time
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

# audio libraries
import librosa

# osprey package
from osprey import (
    clean_row,
    get_audio,
    get_mel,
)

In [ ]:

# setup directory structure
save_folder = f"{save_folder}-{height}-{width}"
collection_map = {
    "iNat": "birdclef-2026/train_audio",
    "XC": "birdclef-2026/train_audio",
    "esc": "ESC-50-master/audio",
    "arca23k": "ARCA23K/ARCA23K.audio/ARCA23K.audio",
    "urban8k": "UrbanSound8K/audio/foldall",
    "dbr": "dbr-dataset/dataset/dograin",
    "freesound": "freesound/outside",
    "random_noise": "random-noise",
}

os.makedirs(save_folder, exist_ok=True)
for key, value in collection_map.items():
    os.makedirs(f"{save_folder}/{value}", exist_ok=True)

table = pd.read_csv(table_path)
table = table[table.collection.isin(collection_map.keys())]
table.reset_index(inplace=True, drop=True)
primary_labels = table['filename'].apply(
    lambda x: x.split('/')[0] if x.find('/') > 0 else '' 
)
primary_labels = set(primary_labels)
for _ in primary_labels:
    os.makedirs(f"{save_folder}/{collection_map['XC']}/{_}", exist_ok=True)
for _ in primary_labels:
    os.makedirs(f"{save_folder}/{collection_map['iNat']}/{_}", exist_ok=True)

In [ ]:
num = table.shape[0]
curr_time = time.time()
for _ in range(num):
    if _ % 2000 == 0:
        print(f'{_/num*100:.2f}%')
    row = table.iloc[_]
    row = clean_row(row,)
    y, _ = get_audio(
        row, 
        sr=sr, 
        base_folder=base_folder, 
        collection_map=collection_map,
        duration_seconds=duration,
    )
    x, h = get_mel(
        y, 
        sr=sr, 
        height=height, 
        width=width,
        fmin=fmin,
        fmax=fmax,
        duration=duration,
    )
    x_max = x.max()
    x_min = x.min()
    z = (x - x_min) / (x_max - x_min + 1e-6)
    z_uint8 = (z * 255).astype(np.uint8)
    foo = collection_map[row['collection']]
    bar = row['filename'].split('.')
    bar = ''.join(bar[:-1])
    st = row['start_seconds']
    en = row['end_seconds']
    bar += f'-{st}s-{en}.npz'
    np.savez_compressed(
        f"{save_folder}/{foo}/{bar}",
        spectrogram = z,
        audio = f"{row['filename']}",
        collection = f"{row['collection']}",
        start = st,
        end = en,
    )
print(f"Spectrograms computed in {(time.time() - curr_time) / 60 :.2f} minutes")